# Data and model loading: a small alignment tutorial

This notebook loads the `three0` neural raster, the matching Talia image set, and a frozen DINOv3 backbone. The important step is **stimulus alignment**: the neural array and `ImageFolder` do not initially share an image axis order.

By the end, one dataloader batch contains model-ready images and neural targets referring to exactly the same stimuli. Nothing is trained.

## The alignment contract

| Object | Shape/order | Meaning |
|---|---|---|
| Neural raster | `[neurons, time, neural_image]` | Last axis follows the neural-file image order |
| `ImageFolder` | a list in ANN order | Images are sorted by class and filename |
| `idx_ord` | `[neural_image]` | `idx_ord[j]` is the ANN index for neural image `j` |
| Aligned image dataset | `ImageFolder[idx_ord]` | Now follows the neural image order |

Equivalently, precomputed ANN features shaped `[features, ann_image]` are aligned with `features[:, idx_ord]`.

In [ ]:
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path

import h5py
import numpy as np
import torch
import yaml
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.datasets import ImageFolder
from transformers import AutoImageProcessor

# Locate the repository whether Jupyter starts at the root or in scripts/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [cwd, *cwd.parents] if (path / "config.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project config.yaml.")
# end if PROJECT_ROOT is None

ENV = os.getenv("MY_ENV", "tiziano_mac_mini")
with open(PROJECT_ROOT / "config.yaml", "r") as f:
    config = yaml.safe_load(f)
# end with open config
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])

from IT_recap.hf_feature_extraction import (  # noqa: E402
    ProcessorTransform,
    is_valid_image_file,
)
from project_specific_utils.dataloader import (  # noqa: E402
    decode_matlab_strings,
    load_img_natraster,
    map_image_order_from_ann_to_monkey,
    rename_talia_dataset,
)
from useful_stuff.image_processing.computational_models import imgANN  # noqa: E402

In [ ]:
@dataclass
class Cfg:
    # Neural data.
    monkey_name: str = "three0"
    date: str = "250313"
    brain_area: str = "AIT"
    new_fs: int = 100

    # Stimuli and frozen image model.
    folder_name: str = "talia_20each_tizi"
    model_name: str = "dino_v3_l"
    model_source: str = "facebook/dinov3-vitl16-pretrain-lvd1689m"
    img_size: int = 224
    pooling: str = "mean"
    layer_names: list[str] = field(
        default_factory=lambda: ["layer.13.mlp.down_proj"]
    )
    batch_size: int = 2


cfg = Cfg()
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"environment: {ENV}; device: {device}")
cfg

## 1. Load the neural data

`load_img_natraster` reads the MATLAB raster, changes its axes to `[neurons, time, images]`, optionally keeps one brain area, and resamples time. The image axis is already unique for this `natraster` file, so we must preserve its order.

In [ ]:
raster = load_img_natraster(
    paths=paths,
    monkey_name=cfg.monkey_name,
    date=cfg.date,
    new_fs=cfg.new_fs,
    brain_area=cfg.brain_area,
)
neural_array = raster.get_array()

if neural_array.ndim != 3:
    raise ValueError(
        f"Expected [neurons, time, images], received {neural_array.shape}."
    )
# end if neural array has unexpected axes

print(f"neural raster [neurons, time, images]: {neural_array.shape}")
print(f"sampling frequency: {raster.fs} Hz")

## 2. Load the model preprocessing and image dataset

The Hugging Face image processor is part of the checkpoint: it fixes resizing, normalization, and tensor conversion. `ImageFolder` then determines the order in which the ANN would see the stimuli. The first run may download checkpoint files.

In [ ]:
image_processor = AutoImageProcessor.from_pretrained(
    cfg.model_source,
    use_fast=True,
)
stimuli_root = Path(paths["livingstone_lab"]) / "Stimuli" / cfg.folder_name
image_dataset = ImageFolder(
    root=stimuli_root,
    transform=ProcessorTransform(image_processor),
    is_valid_file=is_valid_image_file,
    allow_empty=True,
)
ann_image_names = [Path(path).name for path, _ in image_dataset.samples]

sample_image, _ = image_dataset[0]
if tuple(sample_image.shape[-2:]) != (cfg.img_size, cfg.img_size):
    raise ValueError(
        f"Processor returned {tuple(sample_image.shape[-2:])}, "
        f"expected {(cfg.img_size, cfg.img_size)}."
    )
# end if processor output size is unexpected

print(f"ImageFolder images: {len(image_dataset)}")
print(f"processed image [channels, height, width]: {tuple(sample_image.shape)}")
print("first three ANN-order names:", ann_image_names[:3])

## 3. Compute and verify the stimulus mapping

The project helper returns integer indices that select ANN-order images in neural order. We also decode the MATLAB names here so the mapping is visible rather than trusted implicitly. For this renamed Talia folder, spaces are removed and an underscore is inserted before the numeric suffix.

In [ ]:
idx_ord = np.asarray(
    map_image_order_from_ann_to_monkey(
        paths, cfg.monkey_name, cfg.date, image_dataset
    ),
    dtype=int,
)

# Decode the neural-file names only to make the helper's assertion inspectable.
allimages_path = (
    Path(paths["data_path"])
    / "data"
    / f"{cfg.monkey_name}_allimages{cfg.date}.mat"
)
with h5py.File(allimages_path, "r") as h5file:
    key = "allimages" if "allimages" in h5file else "uniqueImage"
    neural_image_names = sorted(
        set(decode_matlab_strings(h5file, h5file[key][:]))
    )
# end with neural image-name file
if stimuli_root.name == "talia_20each_tizi":
    neural_image_names = rename_talia_dataset(neural_image_names)
# end if Talia filenames were renamed

aligned_ann_names = [ann_image_names[ann_idx] for ann_idx in idx_ord]
assert aligned_ann_names == neural_image_names
assert len(idx_ord) == neural_array.shape[2]
assert len(np.unique(idx_ord)) == len(idx_ord)

print(f"mapping length: {len(idx_ord)} unique indices")
for neural_idx in range(3):
    ann_idx = idx_ord[neural_idx]
    print(
        f"neural index {neural_idx:>3} <- ANN index {ann_idx:>3}: "
        f"{ann_image_names[ann_idx]}"
    )
# end for first aligned stimuli

The direction of `idx_ord` is worth making explicit:

```python
aligned_images = Subset(image_dataset, idx_ord)
aligned_precomputed_features = ann_features[:, idx_ord]
```

Do **not** apply `idx_ord` to the neural raster: it is already in the target order.

## 4. Load the frozen model

`imgANN` wraps the pretrained backbone and forward hooks. We request one layer to keep this demonstration small. Mean pooling converts its token output into one vector per image.

In [ ]:
ann = imgANN(
    model_name=cfg.model_name,
    pkg="hf",
    img_size=cfg.img_size,
    relevant_layers=cfg.layer_names,
    pooling=cfg.pooling,
    dtype=torch.float32,
    repo_url=cfg.model_source,
    device=device,
)
ann.create_forward_hook(cfg.layer_names)
print(ann)
print("hooked layers:", list(ann.get_handles()))

## 5. Pair aligned images with neural targets

After reordering the images, both objects use image index `i` for the same stimulus. The dataset below changes neural axes once, from `[neurons, time, images]` to the batch-friendly `[images, time, neurons]`.

In [ ]:
class AlignedImageNeuralDataset(Dataset):
    """Pair model-ready images with same-stimulus dynamic neural targets."""

    def __init__(self, aligned_images, neural_activity):
        # Convert [neurons, time, images] to [images, time, neurons].
        neural_targets = np.asarray(neural_activity).transpose(2, 1, 0)
        if len(aligned_images) != neural_targets.shape[0]:
            raise ValueError(
                f"Image/target counts differ: {len(aligned_images)} and "
                f"{neural_targets.shape[0]}."
            )
        # end if image and target counts differ
        self.aligned_images = aligned_images
        self.neural_targets = torch.as_tensor(
            neural_targets, dtype=torch.float32
        )

    def __len__(self):
        return len(self.aligned_images)
    # EOF

    def __getitem__(self, index):
        image, _ = self.aligned_images[index]
        return image, self.neural_targets[index]
    # EOF
# EOC


aligned_images = Subset(image_dataset, idx_ord.tolist())
paired_dataset = AlignedImageNeuralDataset(aligned_images, neural_array)
loader = DataLoader(
    paired_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    pin_memory=device.type == "cuda",
)
pixel_values, neural_targets = next(iter(loader))

print(f"images [batch, channels, height, width]: {tuple(pixel_values.shape)}")
print(f"targets [batch, time, neurons]: {tuple(neural_targets.shape)}")
print("batch stimulus names:", aligned_ann_names[: cfg.batch_size])

## 6. Run one aligned model batch

Only the images enter DINOv3. The neural batch stays beside the model features as the target for encoding, RSA, or a temporal prediction model.

In [ ]:
with torch.inference_mode():
    model_features = ann.extract_features(
        {"pixel_values": pixel_values.to(device)}
    )
# end with inference mode

for layer_name in cfg.layer_names:
    layer_features = model_features[layer_name]
    if layer_features.shape[0] != neural_targets.shape[0]:
        raise ValueError("Model and neural batch sizes do not match.")
    # end if batch sizes differ
    print(f"{layer_name}: model features {tuple(layer_features.shape)}")
# end for layer
print(f"aligned neural targets: {tuple(neural_targets.shape)}")

# Hooks are no longer needed after this demonstration.
ann.clear_hooks()

## Takeaway

The alignment is purely along the stimulus axis:

1. neural data defines the target image order;
2. `ImageFolder` defines the initial ANN order;
3. `idx_ord` selects ANN images/features in neural order;
4. the paired batch has model input `[B, C, H, W]` and neural target `[B, T, N]`.

Keep `shuffle=False` while inspecting alignment. During training, shuffling the **paired dataset** is safe because each image and target move together.